# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egeklc/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
!git clone https://github.com/egeklc/flyrank-internship.git

import os
os.chdir("flyrank-internship")
print(os.getcwd())

fatal: destination path 'flyrank-internship' already exists and is not an empty directory.
/content/flyrank-internship


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

**Signal 1:** Staleness

Claim: Older pages should receive higher review priority

In [3]:
staleness = (df.groupby(["freshness_tier"])["is_declining"].agg(["count", "mean"])
            .rename(columns={"count":"total",
                             "mean": "decline_rate"})
            .reindex(["0-30", "31-90", "91-180", "181+"])
             )

print(staleness)

                total  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264


Verdict: MIXED

Staleness shows a mixed relationship with decline. The decline rate increases up to the 91-180 day bucket, where it reaches its highest level and then it decreases for the 181+ bucket. Therefore we can't use the staleness as a rule.

**Signal 2:** Position vs. CTR

Claim: CTR falls as average position gets worse

In [4]:
visible = df[df["impressions_90d"] >= 500]
pos = (visible.groupby("position_tier")["ctr"].agg(["count", "mean"])
        .rename(columns={"count": "total",
                         "mean": "mean_ctr"})
        .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"]))
print(pos)

               total  mean_ctr
position_tier                 
top_3            458  0.346572
page_1          7064  0.338808
striking        4485  0.266798
page_3_5        4330  0.143236
deep             389  0.043213


Verdict: CONFIRMED

Among pages meeting the volume floor (`impressions_90d` ≥ 500), the data shows a strong relationship between CTR and average position. Pages with worse positions generally have lower CTR. Therefore position can be used as relevant signal in the baseline.

## 2. Build the ranked queue (writes the CSV)
Code the score, rank everything, write work/outputs/baseline_action_score.csv.

In [5]:
volume_floor = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)].copy()
expected_ctr = volume_floor.groupby("position_tier")["ctr"].median().rename("expected_ctr")
print(expected_ctr)

position_tier
deep        0.00
page_1      0.24
page_3_5    0.09
striking    0.17
top_3       0.20
Name: expected_ctr, dtype: float64


In [6]:
baseline = df.copy()

baseline["visible"] = (baseline["impressions_90d"] >= 500).astype(int)
baseline["stale"] = (baseline["days_since_last_update"] >= 91).astype(int)

baseline["expected_ctr"] = (baseline["position_tier"].map(expected_ctr))
baseline["ctr_gap"] = (baseline["expected_ctr"] - baseline["ctr"])
baseline["ctr_gap"] = baseline["ctr_gap"].clip(lower=0)

baseline.loc[baseline["avg_position"] <= 0, "ctr_gap"] = 0

In [7]:
baseline["baseline_score"] = (baseline["stale"] * (baseline["impressions_90d"]/baseline["impressions_90d"].max()) * baseline["ctr_gap"])

In [8]:
def get_reason_code(row):
    if row["stale"] and row["visible"] and row["ctr_gap"] > 0:
        return "stale_visible_low_ctr"

    elif row["stale"] and row["visible"]:
        return "stale_visible"

    elif row["ctr_gap"] > 0:
        return "low_ctr_for_position"

    else:
        return "no_strong_signal"


baseline["reason_code"] = baseline.apply(get_reason_code, axis=1)

def get_action_code(row):
    if row["stale"] and row["visible"] and row["ctr_gap"] > 0:
        return "review_content_and_search_performance"

    elif row["stale"] and row["visible"]:
        return "review_content_freshness"

    elif row["ctr_gap"] > 0:
        return "review_search_performance"

    else:
        return "no_immediate_review"


baseline["action_code"] = baseline.apply(get_action_code, axis=1)

In [9]:
ranked_df = baseline.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)


In [10]:
print(ranked_df["action_code"].value_counts())

action_code
review_search_performance                13409
no_immediate_review                      10016
review_content_freshness                  3348
review_content_and_search_performance     3227
Name: count, dtype: int64


In [11]:
output_path = "work/outputs/baseline_action_score.csv"
os.makedirs("work/outputs", exist_ok=True)
ranked_df.to_csv(output_path, index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
cols =["content_id",
        "baseline_score",
        "impressions_90d",
        "avg_position",
        "ctr",
        "expected_ctr",
        "ctr_gap",
        "reason_code",
        "action_code"
        ]
top_20 = ranked_df[cols].head(20)
display(top_20)  # top 20 queue

,content_id,baseline_score,impressions_90d,avg_position,ctr,expected_ctr,ctr_gap,reason_code,action_code
0,content_36ff89c8214e,0.108300,295097,7.3,0.05,0.24,0.19,stale_visible_low_ctr,review_content_and_search_performance
1,content_5fe46e04994d,0.100000,517715,4.2,0.14,0.24,0.10,stale_visible_low_ctr,review_content_and_search_performance
2,content_c8e9d6ab9013,0.096738,208678,9.7,0.00,0.24,0.24,stale_visible_low_ctr,review_content_and_search_performance
3,content_91652435f57a,0.055487,159590,7.8,0.06,0.24,0.18,stale_visible_low_ctr,review_content_and_search_performance
4,content_c1fe78bc4e37,0.054377,134055,7.5,0.03,0.24,0.21,stale_visible_low_ctr,review_content_and_search_performance
5,content_a7427266c305,0.050500,201111,5.7,0.11,0.24,0.13,stale_visible_low_ctr,review_content_and_search_performance
6,content_b115f7c74779,0.050083,123469,8.0,0.03,0.24,0.21,stale_visible_low_ctr,review_content_and_search_performance
7,content_97a86caf3a3d,0.048490,147670,6.4,0.07,0.24,0.17,stale_visible_low_ctr,review_content_and_search_performance
8,content_cb112fce36be,0.047889,309910,5.6,0.16,0.24,0.08,stale_visible_low_ctr,review_content_and_search_performance
9,content_4a6607efcb46,0.047001,128068,2.2,0.01,0.20,0.19,stale_visible_low_ctr,review_content_and_search_performance


In [13]:
def why_wrong(r):
    if r["avg_position"] <= 3:
        return "Low CTR may be explained by search intent or search-result characteristics despite the strong position."
    elif r["ctr"] == 0:
        return "Zero CTR may be explained by search intent or search-result characteristics rather than content performance."
    elif r["impressions_90d"] > 100000:
        return "The high impression volume may drive the score even if the CTR gap is relatively small."
    elif r["ctr_gap"] < 0.10:
        return "The ranking may be influenced more by impression volume than by the relatively small CTR gap."
    else:
        return "The position-based CTR benchmark may not accurately represent this page's actual search queries."

top_20["why_wrong"] = top_20.apply(why_wrong, axis=1)
for _, r in top_20.iterrows():
    print(f"{r["content_id"]}  --> {r["action_code"]}")
    print(f"why: {r["impressions_90d"]} impressions at pos {r["avg_position"]}, CTR {r["ctr"]} vs expected CTR {r["expected_ctr"]} (gap {r["ctr_gap"]})")
    print(f"wrong: {r["why_wrong"]}")
    print("---------------------------------------")

content_36ff89c8214e  --> review_content_and_search_performance
why: 295097 impressions at pos 7.3, CTR 0.05 vs expected CTR 0.24 (gap 0.19)
wrong: The high impression volume may drive the score even if the CTR gap is relatively small.
---------------------------------------
content_5fe46e04994d  --> review_content_and_search_performance
why: 517715 impressions at pos 4.2, CTR 0.14 vs expected CTR 0.24 (gap 0.09999999999999998)
wrong: The high impression volume may drive the score even if the CTR gap is relatively small.
---------------------------------------
content_c8e9d6ab9013  --> review_content_and_search_performance
why: 208678 impressions at pos 9.7, CTR 0.0 vs expected CTR 0.24 (gap 0.24)
wrong: Zero CTR may be explained by search intent or search-result characteristics rather than content performance.
---------------------------------------
content_91652435f57a  --> review_content_and_search_performance
why: 159590 impressions at pos 7.8, CTR 0.06 vs expected CTR 0.24 (gap 0.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks near the top are pages with 0.00 CTR, where the large CTR gap can inflate the score and treat them to be strong content opportunities when they may not actually be.


Leakage check:

`trend_direction`,`trend_pct` and `is_declining` are not used as features.
The baseline uses historical `impressions_90d`, `ctr`, `avg_position`, and `days_since_last_update`.
No future reporting window is used in the baseline features.
`content_id` is only an identifier and is not used as a feature.

In [14]:
weak_picks = top_20[top_20["ctr"] == 0]
for _, row in weak_picks.iterrows():
    print(f"Weak pick in the top 20 (where CTR = 0): {row["content_id"]}")

Weak pick in the top 20 (where CTR = 0): content_c8e9d6ab9013


## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.